# Glacier modelling inputs

In [ ]:
# geospatial
import geopandas as gpd
import pandas as pd

# basic
import os
from glob import glob
from tqdm import tqdm

# oggm
from oggm import cfg, workflow, utils
from MBsandbox.mbmod_daily_oneflowline import process_custom_daily_data
from MBsandbox.mbmod_daily_oneflowline import process_gcm_data

from src.utils import load_config
config = load_config()

In [ ]:
cfg.initialize(logging_level='ERROR', future = True)

cfg.PARAMS['use_multiprocessing']  = True
cfg.PARAMS['baseline_climate']     = ''
cfg.PARAMS['prcp_scaling_factor']  = 1
cfg.PARAMS['hydro_month_sh']       = 1
cfg.PARAMS['hydro_month_nh']       = 1
cfg.PARAMS['border']               = 160 
cfg.PARAMS['geodetic_mb_period']   = '2000-01-01_2020-01-01'
cfg.PARAMS['store_model_geometry'] = True
cfg.PARAMS['continue_on_error']    = True
cfg.PARAMS['rgi_version'] = 62

cfg.PATHS['working_dir']  = config["dir"]["pipeline"] + "/oggm/"
cfg.PATHS['climate_path'] = config["dir"]["pipeline"] + "/oggm/OGGM_climate_historical.nc"

## Load outlines

In [ ]:
rgi_16 = utils.get_rgi_region_file(16, version='62') # Low latitudes
rgi_17 = utils.get_rgi_region_file(17, version='62') # Southern Andes
rgi_16 = gpd.read_file(rgi_16)
rgi_17 = gpd.read_file(rgi_17)

rgi_id = pd.concat([rgi_16, rgi_17])
rgi_id = rgi_id[(rgi_id.CenLon < config["area"]["max_lon"]) & 
                (rgi_id.CenLon > config["area"]["min_lon"]) & 
                (rgi_id.CenLat < config["area"]["max_lat"])]
#rgi_id['Name'] = rgi_id['Name'].fillna('')

In [ ]:
base_url = 'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.4/L1-L2_files/elev_bands/'
gdirs    = workflow.init_glacier_directories(rgi_id, from_prepro_level=2, prepro_border = 160, prepro_base_url = base_url,
                                           reset = True, force = True)  # force: no y/n prompt, so it also runs with nbconvert

## Load climate

In [ ]:
# historical climate (ERA5).ERA5 corrected with CHELSA V2.1, interpolated to the GCM grid
from src.glaciers import create_OGGM_climate

create_OGGM_climate(prcp = config["dir"]["data"] + "/" + config["climate"]["prcp_file"],
                    temp = config["dir"]["data"] + "/era5/tas_ERA5_1960_2025.nc",
                    hgt  = config["dir"]["data"] + "/dem/DEM_Andes_270m.tif",
                    y0 = int(config["time"]["ref_start"][0:4]), 
                    y1 = int(config["time"]["ref_end"][0:4]),
                    output_path = cfg.PATHS['climate_path'],
                    
                    # same grid as the GCMs, so that historical and future climate share the reference height
                    grid = sorted(glob(config["dir"]["data"] + "/" + config["climate"]["cmip6_dir"] + "/*.nc"))[0],
                    hgt_cell_mean = True)

In [ ]:
workflow.execute_entity_task(process_custom_daily_data, gdirs,
                             temporal_resol='daily',
                             path = cfg.PATHS['climate_path'],
                             y0 = int(config["time"]["ref_start"][0:4]), 
                             y1 = int(config["time"]["ref_end"][0:4]),
                             output_filesuffix="_ERA5");

In [ ]:
# Future scenarios: GDPCIR files with the same precipitation correction as ERA5
# (config.yml:climate.cmip6_dir, written by AndeanGC tools/correct_era5_chelsa.py)
scenarios  = glob(config["dir"]["data"] + "/" + config["climate"]["cmip6_dir"] + "/*.nc")

for gcm in tqdm(scenarios):    

        rid = os.path.splitext(os.path.basename(gcm))[0].rsplit('_', 2)[0]

        # process future climate data
        workflow.execute_entity_task(process_gcm_data, gdirs, temporal_resol='daily',
                                     path_climate = gcm,
                                     path_hgt  = cfg.PATHS['climate_path'] ,
                                     y0 = int(config["time"]["future_start"][0:4]), 
                                     y1 = int(config["time"]["future_end"][0:4]),
                                     output_filesuffix= "_daily_" + rid);